# Phase 4 Prototype — Langfuse (Production Tracing)

**Status: exploratory prototype, not a Phase 1 implementation item.** Per `CLAUDE.md`, Langfuse is a *trigger-based addition* — it is only adopted "once production tracing, annotation workflows, and prompt release management become real operational requirements" (Phase 4). That trigger has not fired in this repo: there is no production traffic yet.

This notebook exists to answer one question for a stakeholder demo: **"what would Phase 4 look like?"** — not to stand up a real production system. It self-hosts Langfuse, replays this repo's *existing* golden-set eval run (same data as `model_eval_deepeval_mlflow.ipynb`) through it as synthetic "production" traffic, attaches the DeepEval scores as Langfuse trace scores, and shows what the resulting trace/session/score UI looks like.

> **This does not change the binding architecture.** MLflow remains the system of record for offline/pre-release truth (experiments, golden-dataset runs, CI gate results). Nothing here is wired into the CLI runner, CI, or any production path. If Phase 4 is later triggered for real, this notebook is a starting point, not the implementation.

## Table of contents

1. [Requirements](#1-requirements)
2. [Configuration](#2-configuration)
3. [Self-host Langfuse (Docker Compose)](#3-self-host-langfuse-docker-compose)
4. [Connect the SDK](#4-connect-the-sdk)
5. [Replay golden-set cases as traces](#5-replay-golden-set-cases-as-traces)
6. [Attach DeepEval scores to traces](#6-attach-deepeval-scores-to-traces)
7. [What to look at in the UI](#7-what-to-look-at-in-the-ui)
8. [Feature notes: Langfuse vs. this repo's MLflow-based Phase 1](#8-feature-notes-langfuse-vs-this-repos-mlflow-based-phase-1)

## 1. Requirements

- Docker + Docker Compose, for self-hosting Langfuse (step 3).
- `pip install -r requirements.txt` (includes `langfuse`).
- A prior run of `model_eval_deepeval_mlflow.ipynb` (or `model_eval_deepeval.ipynb`) with a saved JSON artifact under `notebooks/artifacts/deepeval-model-eval/` — this notebook replays that run rather than re-querying a model, so it reuses real scores instead of fabricating them.

In [ ]:
%pip install -q -r requirements.txt

## 2. Configuration

Langfuse connection details and the path to the eval artifact to replay. Following this repo's env-var-first convention (`CLAUDE.md`): every value below reads from an env var first, with an explicit placeholder fallback so a missing var fails loudly instead of silently.

In [ ]:
import json
import os
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "notebooks" / "sample-prompts").exists() else Path.cwd()
assert (REPO_ROOT / "notebooks" / "sample-prompts").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)

# ---- Langfuse connection (self-hosted instance from Section 3) -----------
LANGFUSE_HOST = os.environ.get("LANGFUSE_HOST", "http://localhost:3000")
LANGFUSE_PUBLIC_KEY = os.environ.get("LANGFUSE_PUBLIC_KEY", "")  # pk-lf-... , from the Langfuse UI project settings
LANGFUSE_SECRET_KEY = os.environ.get("LANGFUSE_SECRET_KEY", "")  # sk-lf-...

# ---- Eval artifact to replay as "production" traffic ---------------------
# Any results_*.json produced by model_eval_deepeval_mlflow.ipynb or
# model_eval_deepeval.ipynb. Pick the most recent one by default.
ARTIFACT_DIR = REPO_ROOT / "notebooks" / "artifacts" / "deepeval-model-eval"
_candidates = sorted(ARTIFACT_DIR.glob("results_*.json"), key=lambda p: p.stat().st_mtime, reverse=True) if ARTIFACT_DIR.exists() else []
ARTIFACT_PATH = _candidates[0] if _candidates else None

assert ARTIFACT_PATH is not None, (
    f"No results_*.json found under {ARTIFACT_DIR}. Run model_eval_deepeval_mlflow.ipynb "
    "(or model_eval_deepeval.ipynb) first, then re-run this notebook."
)
print(f"Langfuse host:   {LANGFUSE_HOST}")
print(f"Replaying:       {ARTIFACT_PATH}")

## 3. Self-host Langfuse (Docker Compose)

Langfuse's self-hosted stack is heavier than this repo's current MLflow-only backend: web UI + async worker + Postgres (metadata) + ClickHouse (trace analytics) + Redis (queueing) + MinIO (blob storage) — six containers total. Run this **once**, outside the notebook, in a terminal:

```bash
git clone https://github.com/langfuse/langfuse.git langfuse-selfhost
cd langfuse-selfhost
docker compose up -d
```

Then:

1. Open `http://localhost:3000`, create an account (local-only — nothing leaves the machine) and a project.
2. In the project's **Settings → API Keys**, create a keypair and copy the public/secret key into Section 2 above (or export `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` before starting Jupyter).

> **Air-gap note**: unlike this repo's other tools, self-hosted Langfuse core does not phone home by default (per its own docs) and is deployable in an air-gapped cluster. No additional telemetry env var is required here, but verify against the installed version's docs before relying on that for a real deployment.

## 4. Connect the SDK

`get_client()` reads `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` / `LANGFUSE_HOST` from the environment. We export the values from Section 2 first so this works whether they came from the notebook config or from the shell.

In [ ]:
assert LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY, (
    "Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY in Section 2 "
    "(from the Langfuse UI → Settings → API Keys, per Section 3)."
)
os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
os.environ["LANGFUSE_HOST"] = LANGFUSE_HOST

from langfuse import get_client

langfuse = get_client()

# Preflight: confirm the self-hosted instance is actually reachable before we
# spend a whole replay run against it (mirrors the judge preflight pattern in
# model_eval_deepeval_mlflow.ipynb).
if not langfuse.auth_check():
    raise RuntimeError(
        f"Langfuse auth check FAILED against {LANGFUSE_HOST}. "
        "Confirm the containers from Section 3 are running (`docker compose ps`) "
        "and that the public/secret key in Section 2 match the project you created."
    )
print(f"OK — authenticated against {LANGFUSE_HOST}.")

## 5. Replay golden-set cases as traces

Load the saved eval artifact (input / actual_output / expected_output per case) and log each case as a Langfuse trace with a nested `generation` span, as if it were a live request. This is the same shape a real production integration would produce (e.g. wrapping the model-under-test call in `model_eval_deepeval_mlflow.ipynb`'s Section 4 with `@observe`), just replayed after the fact from saved data instead of captured live.

In [ ]:
with open(ARTIFACT_PATH, encoding="utf-8") as f:
    artifact = json.load(f)

model_name = artifact.get("model", "unknown-model")
cases = artifact["cases"]  # [{input, expected_output, actual_output}, ...]

trace_ids = []  # keep (case_index -> trace_id) so Section 6 can attach scores

for i, case in enumerate(cases):
    with langfuse.start_as_current_observation(
        as_type="span", name="golden-set-replay",
        input={"prompt": case["input"]},
    ) as span:
        span.update_trace(
            name=f"qa-case-{i}",
            tags=["phase4-prototype", "replayed", model_name],
            metadata={"source_artifact": str(ARTIFACT_PATH.name), "case_index": i},
        )
        with langfuse.start_as_current_observation(
            as_type="generation", name="model-under-test",
            model=model_name,
            input=[{"role": "user", "content": case["input"]}],
        ) as generation:
            generation.update(output=case["actual_output"])
        span.update(output=case["actual_output"])
        trace_ids.append(span.trace_id)

langfuse.flush()
print(f"Replayed {len(cases)} cases as Langfuse traces (model={model_name}).")

## 6. Attach DeepEval scores to traces

The eval artifact already has real DeepEval judge scores (Correctness, Answer Relevancy) from `model_eval_deepeval_mlflow.ipynb`. Attach them as Langfuse trace scores via `create_score` — this is what "annotate live traces with quality scores" looks like once a real judge-on-production pipeline exists. **These are the same scores already logged to MLflow for this run** — nothing new is computed here, they are just projected onto the replayed traces to show what the Langfuse scoring UI looks like.

In [ ]:
rows = artifact["results"]  # [{case, metric, score, passed, reason}, ...] from the DeepEval run

n_scores = 0
for row in rows:
    if row["score"] is None:
        continue
    trace_id = trace_ids[row["case"]]
    langfuse.create_score(
        trace_id=trace_id,
        name=row["metric"],           # "Correctness" / "Answer Relevancy" — matches docs/metric-registry.md naming
        value=row["score"],
        data_type="NUMERIC",
        comment=row["reason"],
    )
    n_scores += 1

langfuse.flush()
print(f"Attached {n_scores} DeepEval scores to {len(trace_ids)} replayed traces.")

## 7. What to look at in the UI

Open `http://localhost:3000` (or `LANGFUSE_HOST`) and check:

- **Traces** — each replayed case as a trace with a nested `model-under-test` generation span; click one to see input/output and latency.
- **Scores** (on a trace's detail page, or the Scores tab) — the Correctness / Answer Relevancy values from Section 6, with the judge's reason as the comment.
- **Sessions/Tags** — traces are tagged `phase4-prototype`, `replayed`, and the model name, so they can be filtered separately from anything else in the instance.
- **Annotation queues** (Settings → Annotation Queues) — not populated here, but this is where a human reviewer would spot-check a sample of production traces; relevant once real traffic exists.
- **Prompt management** (Prompts tab) — not exercised by this notebook; this is where prompt *release* labels (prod/staging) would live per `CLAUDE.md`'s system-of-record split, distinct from MLflow's prompt *development* versions.

## 8. Feature notes: Langfuse vs. this repo's MLflow-based Phase 1

- **Tracing is native and UI-first.** MLflow Tracing exists but the UI/workflow in this repo currently stops at run-level params/metrics (`model_eval_deepeval_mlflow.ipynb` Section 8); Langfuse's whole UI is built around the trace/span/session view.
- **Scores are a first-class object attached to traces**, not just run metrics — `create_score` on a `trace_id` is the online-truth equivalent of `log_eval_run`'s MLflow metrics for offline truth. This is exactly the split `CLAUDE.md` draws: MLflow owns the offline run this data originally came from; if this were real production traffic, Langfuse would be the *only* place these scores live.
- **Annotation queues and prompt release labels** have no equivalent in the current Phase 1 stack — MLflow's prompt registry covers *development* versions/lineage, not *release* (prod/staging) labels.
- **Heavier self-hosted footprint**: Postgres + ClickHouse + Redis + MinIO + two app containers (web, worker), vs. MLflow's single backend + tracking DB + artifact store. Same tradeoff noted for Opik in `opik/README.md`.
- **Judge scores still need to come from somewhere** — Langfuse can run "managed LLM-as-judge" evaluation on traces directly, which this notebook did *not* exercise (it replayed pre-computed DeepEval scores instead). A real Phase 4 adoption would need to decide whether production scoring uses Langfuse's own judge pipeline or forwards to the same DeepEval judge configuration used offline — per the pinned-judge rule, mixing judges between offline and online would make the two numbers incomparable.

**Bottom line**: this confirms Langfuse's trace+score model is a plausible fit for the *online/post-release* half of `CLAUDE.md`'s system-of-record split, and that wiring an existing DeepEval-scored run into it is mechanically simple (Sections 5–6 are ~30 lines). It does not by itself justify triggering Phase 4 — that still requires real production traffic and an operational need for tracing/annotation/release-management, per the entry criterion in `CLAUDE.md`.